# 9장 — 4비트까지: 블록 양자화와 GGUF 🧱

웹 챕터의 아웃라이어 전쟁을 코드로 재현하고, 블록 크기의 트레이드오프를 스윕하고,
GGUF 파일 영수증(실효 비트 → GB)을 계산합니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 4GB 파일의 완성된 영수증.
**"블록별 축척(분단별 채점)이 눈금 15개짜리 int4 를 실용으로 만든다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch9')

## 1. 아웃라이어 전쟁: 통짜 vs 블록 (int4)

In [ ]:
import numpy as np

def quant_err(w, block_size, bits=4):
    """block_size 단위로 축척을 따로 두는 대칭 양자화의 복원 오차 배열."""
    levels = 2 ** (bits - 1) - 1
    errs = np.empty_like(w)
    for b in range(0, len(w), block_size):
        blk = w[b:b + block_size]
        s = np.abs(blk).max() / levels
        errs[b:b + block_size] = np.abs(np.round(blk / s) * s - blk)
    return errs

rng = np.random.default_rng(9)
w = rng.normal(0, 0.03, 128)
w[0] = 0.5                                    # 🦹 아웃라이어 침입 (원래 max ≈ 0.09)

err_mono = quant_err(w, 128)                  # 통짜 — 축척 하나
err_block = quant_err(w, 32)                  # 블록 — 32개마다 축척

m_mono = err_mono[1:].mean()                  # 무고한 127명의 평균 오차
m_block = err_block[1:].mean()
print(f'통짜  평균 오차: {m_mono:.5f}')
print(f'블록  평균 오차: {m_block:.5f}   (통짜의 1/{m_mono / m_block:.1f})')

assert m_mono > m_block * 2.5, '블록이 무고한 127명의 오차를 수 배 줄인다 — 피해 국한'
assert err_block[1:32].mean() > err_block[32:].mean() * 2, '피해는 아웃라이어의 블록(1분단) 안에만'
print()
print('✔ 확인: 값은 그대로, 채점 단위만 줄였다. 분단별 채점이 int4 를 실용으로')

## 2. 블록 크기 스윕: 정확도와 오버헤드의 다이얼

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

sizes = [8, 16, 32, 64, 128]
errs = [quant_err(w, bs)[1:].mean() for bs in sizes]
overhead = [16 / bs for bs in sizes]          # 블록당 fp16 축척 → 파라미터당 추가 비트

fig, ax1 = plt.subplots(figsize=(6.5, 3.8))
ax1.plot(sizes, errs, 'o-', color='tab:red', label='무고한 127명의 평균 오차')
ax1.set_xlabel('블록 크기'); ax1.set_ylabel('평균 오차', color='tab:red'); ax1.set_xscale('log', base=2)
ax2 = ax1.twinx()
ax2.plot(sizes, overhead, 's--', color='tab:blue', label='축척 오버헤드 (비트/파라미터)')
ax2.set_ylabel('추가 비트/파라미터', color='tab:blue')
ax1.set_title('블록 크기 다이얼: 작을수록 정확하지만 비싸다'); fig.tight_layout(); plt.show()

assert errs[0] < errs[-1], '블록이 작을수록 오차↓ (아웃라이어 격리 + 눈금 밀착)'
assert overhead[0] > overhead[-1], '블록이 작을수록 축척 오버헤드↑'
assert 16 / 32 == 0.5, '실전 표준(32개) 기준 +0.5비트/파라미터'
print('✔ 확인: 32~128 이 실전 균형점. K-quants 의 2단 축척은 이 딜레마 자체를 완화하는 설계')

## 3. GGUF 영수증: 실효 비트로 파일 크기 계산

1장의 수수께끼("계산 3.3GB vs 실파일 4GB")를 정산합니다.

In [ ]:
N = 6.57e9                                     # 1장의 7B급 파라미터 수
TAGS = {                                        # 태그: 실효 비트/파라미터 (근사)
    'F16': 16.0,     # 양자화 없음 (7장)
    'Q8_0': 8.5,     # int8 + 블록 축척
    'Q4_K_M': 4.85,  # 4비트 K-quants + M 믹스 — 국민 태그
    'Q4_0': 4.5,     # 4비트 구식 단순 블록
    'Q2_K': 2.6,     # 초압축
}
print(' 태그       실효비트   7B급 파일 크기')
for name, bits in TAGS.items():
    print(f' {name:<9}  {bits:>5}     {N * bits / 8 / 1e9:5.1f} GB')

assert abs(N * 4.85 / 8 / 1e9 - 4.0) < 0.1, 'Q4_K_M ≈ 4.0GB — 1장 수수께끼의 정산 완료'
assert abs(N * 4 / 8 / 1e9 - 3.3) < 0.1, '순수 4비트 이론치는 3.3GB — 차이가 축척·믹스·메타데이터'
assert abs(N * 16 / 8 / 1e9 - 13.1) < 0.1 and abs(N * 8.5 / 8 / 1e9 - 7.0) < 0.1
print()
print('✔ 확인: Ollama 태그 목록이 전부 계산으로 읽힌다. 품질 손실의 측정은 10장(퍼플렉시티)')

**직접 해보기**: ① s1 의 아웃라이어를 0.2, 1.0 으로 바꿔 배율이 어떻게 변하는지 보세요.
② 여러분 PC RAM 에 들어가는 가장 높은 품질의 태그는 무엇인가요? (가중치 몫 기준이고,
문맥 몫까지 정산하는 법은 11장 캡스톤에서.)

## 4. 관찰 정리 + 🤖 AI 연결

- **아웃라이어**(3부 5장 분포의 꼬리)는 축척을 부풀려 **다수의 정밀도를 희생**시킨다.
  int4(눈금 ±7)에선 치명적.
- **블록 양자화** = 분단별 채점(32개마다 축척): 피해 국한 + 눈금 밀착. 가격은
  **+0.5비트/파라미터**(32 기준). 블록 크기는 다이얼(작을수록 정확·비쌈).
- **GGUF 문패**: Q숫자(비트)·K(2단 축척)·S/M/L(중요 텐서 믹스)·_0(구세대).
  실효 비트: Q4_0 4.5 / **Q4_K_M ≈ 4.85 → 7B급 4.0GB**. 1장 영수증 정산 완료.
- **다시 쓰는 수학**: 3부 5장(꼬리), 5부 2장(압축 철학: 드문 것에 비트 더), 8장(축척),
  1장(몸무게 공식).

**다음 챕터**: 10장(품질 성적표). "얼마나 잃었나"를 5부 7장의 퍼플렉시티로 측정한
용량↔품질 트레이드오프 곡선.